# Task 4: Model evaluation, step by step

We will evaluate the loan classifier you trained in Task 3 using its **original test set**. You already have the model and the data. This lesson explains accuracy, precision, recall, F1, confusion matrices, and classification reports.

Run the cells from top to bottom using the existing `.venv` Python kernel. The saved model is used for prediction only. This is a closer look at Task 3's existing evaluation, using the same 123 applications.

## 1. Load the tools and find the project

**Training** means learning from examples. **Evaluation** means comparing predictions with known answers on examples excluded from training. We must keep the original train/test assignments and reuse the fitted preprocessing.

The helper `load_test_predictions()` checks the data fingerprint and saved split, loads the locally created model, and verifies that its predictions match Task 3. It never calls `fit()`. You can read its code in `src/evaluate.py`.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, confusion_matrix, classification_report,
)

# Works from the repository root, project folder, or notebook folder.
PROJECT_DIR = None
for parent in [Path.cwd(), *Path.cwd().parents]:
    for candidate in [parent, parent / "projects" / "04_model_evaluation"]:
        if (candidate / "src" / "evaluate.py").is_file():
            PROJECT_DIR = candidate
            break
    if PROJECT_DIR is not None:
        break
if PROJECT_DIR is None:
    raise FileNotFoundError("Open this notebook from inside the project repository.")
sys.path.insert(0, str(PROJECT_DIR / "src"))
from evaluate import (
    load_test_predictions, plot_confusion_matrices,
    plot_classification_report, main,
)
print("Project:", PROJECT_DIR)

## 2. Restore the original test predictions

`y_test` holds the actual answers; `y_pred` holds the model's predictions. Both use the same row order. **Not Approved = 0** and **Approved = 1**. Approved is our **positive class**. Positive is a label convention, not a judgment about an applicant.

The baseline always predicts the most frequent **training** label. A saved split manifest tells us exactly which application IDs belong to training or testing. We do not create another random split.

In [ ]:
y_test, y_pred, baseline, predictions, provenance = load_test_predictions()
print("Model:", provenance["selected_model"])
print("Training applications:", provenance["train_rows"])
print("Test applications:", provenance["test_rows"])
print("Reproduces Task 3:", provenance["task3_predictions_reproduced"])
display(predictions.head())
display(predictions["actual"].value_counts().rename("Test support"))

The test set has 85 Approved and 38 Not Approved applications. It is **imbalanced**: one class has more examples. An always-Approved prediction gets 85/123 = 69.11% accuracy while finding none of the Not Approved cases. This is why we need several metrics.

The `probability_approved` column is the model's score for class 1. It is not a measured repayment probability. We use the model's saved `predict()` rule without changing its threshold.

## 3. Calculate accuracy, precision, recall, and F1

| Metric | Question answered |
| --- | --- |
| Accuracy | Out of all test applications, how many labels are correct? |
| Approved precision | Out of the applications predicted Approved, how many actually have an Approved label? |
| Approved recall | Out of the applications actually labeled Approved, how many did the model find? |
| Approved F1 | How do precision and recall combine using their harmonic mean? |
| Balanced accuracy | What is the average recall when each class gets equal weight? |

Use the **actual answers first**, then predictions, in these functions. `pos_label=1` makes the positive-class convention explicit. `zero_division=0` reports zero when a ratio is undefined, such as precision for a class that is never predicted.

In [ ]:
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Approved precision": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
    "Approved recall": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
    "Approved F1": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
    "Balanced accuracy": balanced_accuracy_score(y_test, y_pred),
}
display(pd.Series(metrics, name="Test score").to_frame().style.format("{:.2%}"))

Accuracy is **86.18%**, Approved precision **84.00%**, Approved recall **98.82%**, and Approved F1 **90.81%**. These describe this test set. They do not guarantee the same performance on future applications.

Precision and recall answer different questions: high Approved recall means few actual Approved records are missed, while lower Approved precision means some predicted approvals have a Not Approved label.

## 4. Build and read the confusion matrix

A **confusion matrix** counts the four possible outcomes. Here rows are actual labels, and columns are predictions. We explicitly use the order `[0, 1]` so the layout is predictable.

| Actual / predicted | Not Approved (0) | Approved (1) |
| --- | --- | --- |
| Not Approved (0) | True negative (TN) | False positive (FP) |
| Approved (1) | False negative (FN) | True positive (TP) |

A true outcome is correct; a false outcome is an error. Positive and negative refer to the predicted class.

In [ ]:
LABEL_NAMES = ["Not Approved", "Approved"]
matrix = confusion_matrix(y_test, y_pred, labels=[0, 1])
display(pd.DataFrame(matrix, index=LABEL_NAMES, columns=LABEL_NAMES).rename_axis(
    index="Actual", columns="Predicted"
))
tn, fp, fn, tp = (int(n) for n in matrix.ravel())
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"Correct predictions: {tn + tp} out of {matrix.sum()}")

There are **22 true negatives**, **16 false positives**, **1 false negative**, and **84 true positives**.

For example, the 16 false positives are applications predicted Approved even though their recorded label is Not Approved. These are errors against historical approval decisions; the dataset does not tell us whether these applicants would repay a loan.

In [ ]:
# Reconstruct the metrics from the four counts to connect formulas to the results.
manual = pd.Series({
    "Accuracy": (tp + tn) / (tp + tn + fp + fn),
    "Approved precision": tp / (tp + fp),
    "Approved recall": tp / (tp + fn),
    "Approved F1": 2 * tp / (2 * tp + fp + fn),
    "Balanced accuracy": ((tp / (tp + fn)) + (tn / (tn + fp))) / 2,
})
# These denominators are nonzero for this dataset.
assert np.allclose(manual.to_numpy(), list(metrics.values()))
display(manual.rename("Calculated from counts").to_frame().style.format("{:.2%}"))

## 5. Visualize the confusion matrix

The left chart shows counts. The right chart divides each row by the number of actual applications in that class. Each row totals 100%, allowing us to compare performance when class sizes differ. The diagonal in the normalized chart is each class's recall.

In [ ]:
fig = plot_confusion_matrices(y_test, y_pred)
display(fig)
plt.close(fig)

The model finds **84/85 = 98.82%** of Approved records but only **22/38 = 57.89%** of Not Approved records. The larger Approved class helps the overall accuracy look strong.

Balanced accuracy is the mean of these two recalls: **78.36%**. It gives the two classes equal weight.

## 6. Create and visualize a classification report

A **classification report** calculates precision, recall, and F1 for each class. For the Not Approved row, that class is treated as the class of interest.

- **Support:** the number of actual test examples in that class.
- **Macro average:** the ordinary mean of the class scores, giving each class equal weight.
- **Weighted average:** the mean weighted by support, so the larger class contributes more.

The text report also includes overall accuracy. The heatmap focuses on per-class precision, recall, and F1; support counts appear beside the class names.

In [ ]:
print(classification_report(
    y_test, y_pred, labels=[0, 1], target_names=LABEL_NAMES,
    digits=4, zero_division=0,
))
report = classification_report(
    y_test, y_pred, labels=[0, 1], target_names=LABEL_NAMES,
    output_dict=True, zero_division=0,
)
report_table = pd.DataFrame({
    name: report[name]
    for name in [*LABEL_NAMES, "macro avg", "weighted avg"]
}).T
display(report_table.style.format({
    "precision": "{:.2%}", "recall": "{:.2%}",
    "f1-score": "{:.2%}", "support": "{:.0f}",
}))
fig = plot_classification_report(report)
display(fig)
plt.close(fig)

Not Approved precision is **22/23 = 95.65%**: most predictions of Not Approved are correct. Its recall is **22/38 = 57.89%**: the model identifies only a little over half of all actual Not Approved records. A class can have high precision and much lower recall.

For binary classification with both classes present, macro-average recall equals balanced accuracy. Weighted-average recall equals accuracy here. Other macro and weighted scores answer different questions; check their labels.

## 7. Compare with a simple baseline

A useful model should improve on an easy reference. Our baseline always predicts the most frequent label from the training set. We compare it on the **same test applications**. This comparison describes the already chosen model; we do not use the test results to select a different model.

In [ ]:
comparison = pd.DataFrame({
    "Selected model": metrics,
    "Majority baseline": {
        "Accuracy": accuracy_score(y_test, baseline),
        "Approved precision": precision_score(y_test, baseline, zero_division=0),
        "Approved recall": recall_score(y_test, baseline, zero_division=0),
        "Approved F1": f1_score(y_test, baseline, zero_division=0),
        "Balanced accuracy": balanced_accuracy_score(y_test, baseline),
    },
})
display(comparison.style.format("{:.2%}"))

The baseline gets **69.11% accuracy** and **50.00% balanced accuracy**. It has 100% Approved recall because it predicts Approved for everyone, but zero Not Approved recall. This shows why high recall for a single class is not sufficient evidence of good overall performance.

## 8. Inspect errors and save the results

`correct` compares each prediction with its actual label. Filtering its opposite (`~`) shows the mistakes. We retain IDs and evaluation results without copying applicants' feature values into the report. There are 17 errors in this test set.

Reviewing errors helps describe limitations. If you want to improve the model, use training-only validation for choosing changes and reserve new untouched data for a future final evaluation.

In [ ]:
errors = predictions.loc[~predictions["correct"]].copy()
print("Misclassified applications:", len(errors))
display(errors.head(10))

# Save all metrics, prediction tables, figures, and the written findings.
# This calls the same script entry point used from PowerShell; it never fits a model.
main()
print("Written report:", PROJECT_DIR / "reports" / "findings.md")

## 9. Write your conclusion

**Example conclusion:** On the original 123 test applications, the logistic regression model correctly classified 106, achieving 86.18% accuracy. Approved precision was 84.00%, recall 98.82%, and F1 90.81%. It exceeded the majority baseline, but Not Approved recall was only 57.89%. The confusion matrix contained 22 TN, 16 FP, 1 FN, and 84 TP. The classifier therefore performed unevenly across the two classes.

**Limits:** This is one small test split, already evaluated in Task 3, rather than a new independent sample. Historical approval labels do not establish repayment ability, and aggregate scores do not establish fairness. The dataset's source, license, dates, and sampling are unverified.

**Check your understanding:**

1. Why can an always-Approved classifier get about 69% accuracy?
2. Which cell counts applications predicted Approved with an actual Not Approved label?
3. Why can Not Approved precision be high while its recall is much lower?
4. What does support mean, and why does it affect the weighted average?

**Answers:** 1. Approved is the larger class (85/123). 2. False positives (16). 3. It predicts Not Approved for only 23 records; 22 are correct, but 16 of the 38 actual Not Approved records are missed. 4. Support is the actual class count; larger classes receive more weight.

The script and notebook use the same saved model and split. Generated files are in `reports/`, including the figures and a list of misclassified applications.